This notebook collects all latest data about carceral facilities in California

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

## Base facilities data set from FEMA RAPT

In [2]:
fema_boundaries = gpd.read_file("data_sources/facilities/Prison_Boundaries_RAPT.geojson")
len(fema_boundaries)

6471

In [3]:
fema_boundaries = fema_boundaries[fema_boundaries["STATE"] == "CA"]
fema_boundaries = fema_boundaries[fema_boundaries["STATUS"] == "OPEN"].reset_index(drop=True)
len(fema_boundaries)

357

In [4]:
fema_cols_drop = [
    'FID',
    'OBJECTID',
    'NAICS_CODE',
    'NAICS_DESC',
    'SOURCE',
    'SOURCEDATE',
    'VAL_METHOD',
    'VAL_DATE',
    'ZIP4'
]
fema_boundaries = fema_boundaries.drop(columns=fema_cols_drop)

In [5]:
name_cols = ['NAME', 'ADDRESS', 'CITY']
for col in name_cols:
    fema_boundaries[col] = fema_boundaries[col].str.title()
    
# Update all column names to be lowercase
fema_boundaries.columns = [col.lower() for col in fema_boundaries.columns]

## Add centerpoints

In [6]:
def add_facility_centroids(gdf):
    """
    Calculates the centroid of each facility geometry and adds 
    latitude and longitude columns.
    """
    original_crs = gdf.crs
    
    if original_crs is not None:
        centroids = gdf.to_crs(epsg=3857).geometry.centroid.to_crs(original_crs)
    else:
        centroids = gdf.geometry.centroid
    
    # Extract coordinates
    gdf['longitude'] = centroids.x
    gdf['latitude'] = centroids.y
    
    return gdf

In [7]:
facs_with_centers = add_facility_centroids(fema_boundaries)

## Merge with Census Tract ID

In [8]:
tracts = gpd.read_file("data_sources/facilities/cb_2020_06_tract_500k.zip")

In [9]:
def add_census_tracts(gdf, tracts):
    """
    Performs a spatial join using the facility centroids to identify the 
    Census Tract GEOID for each facility.
    """

    points_gdf = gpd.GeoDataFrame(
        gdf[['facilityid']], 
        geometry=gpd.points_from_xy(gdf.longitude, gdf.latitude),
        crs=gdf.crs
    )

    if points_gdf.crs != tracts.crs:
        tracts = tracts.to_crs(points_gdf.crs)

    joined = gpd.sjoin(points_gdf, tracts[['GEOID', 'geometry']], how='left', predicate='within')

    gdf['tract_geoid'] = gdf['facilityid'].map(joined.set_index('facilityid')['GEOID'])

    return gdf

In [10]:
facs_with_centers = add_census_tracts(facs_with_centers, tracts)

## Calculate capacity % where available

In [11]:
def calculate_capacity_metrics(gdf):
    """
    Cleans -999 values and calculates capacity percentage metrics.
    """
    cols_to_clean = ['population', 'capacity']
    for col in cols_to_clean:
        if col in gdf.columns:
            gdf[col] = gdf[col].replace(-999, np.nan)

    # 2. Calculate current capacity_percent (population / capacity)
    if 'population' in gdf.columns and 'capacity' in gdf.columns:
        gdf['capacity_percent'] = gdf['population'] / gdf['capacity']

    return gdf

In [12]:
facs_with_pop = calculate_capacity_metrics(facs_with_centers)

## Re-type after manual review

In [13]:
# After reviewing the FEMA data set, several facilities had their 'type' manually fixed
manual_type_dict = {
    '10000850':'FEDERAL',
    '10000851':'FEDERAL',
    '10000849':'FEDERAL',
    '10000845':'STATE',
    '10000846':'STATE'
}
facs_with_pop['type'] = facs_with_pop['facilityid'].map(manual_type_dict).fillna(facs_with_pop['type'])

## Add isolation distance to nearest Medical & Emergency Response facility

In [14]:
def add_isolation_distance(gdf, medical_path):
    """
    Adds dist_nearest_medical_mi: great-circle distance in miles to the
    nearest Medical & Emergency Response facility (hospitals, ambulance
    services, fire/EMS stations) from the USGS National Map Structures layer.

    Source data: data_sources/national_map_medical_emergency.csv
    Scraper:     scrapers/fetch_national_map_medical.py
    """
    medical = pd.read_csv(medical_path).dropna(subset=["longitude", "latitude"])
    med_lat = np.radians(medical["latitude"].values)
    med_lon = np.radians(medical["longitude"].values)
    R = 3958.8  # Earth radius in miles

    distances = []
    for _, row in gdf.iterrows():
        if pd.isna(row["longitude"]) or pd.isna(row["latitude"]):
            distances.append(np.nan)
            continue
        lat1 = np.radians(row["latitude"])
        lon1 = np.radians(row["longitude"])
        dlat = med_lat - lat1
        dlon = med_lon - lon1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(med_lat) * np.sin(dlon / 2) ** 2
        dist = 2 * R * np.arcsin(np.sqrt(a))
        distances.append(round(float(dist.min()), 2))

    gdf = gdf.copy()
    gdf["dist_nearest_medical_mi"] = distances
    return gdf


In [15]:
facs_with_pop = add_isolation_distance(
    facs_with_pop,
    "data_sources/national_map_medical_emergency.csv"
)


## Add 2020 Census Urban Area flag

In [16]:
def add_urban_area_flag(gdf, ua_path):
    """
    Adds in_urban_area_2020: True if the facility centroid falls within a
    2020 Census Urban Area boundary (Urbanized Areas ≥50k population and
    Urban Clusters 2,500–49,999 population), False otherwise.

    Source: Census Bureau 2020 Urban Area cartographic boundary file
            cb_2020_us_ua20_500k.zip
    """
    ua = gpd.read_file(ua_path).to_crs(gdf.crs)
    points = gpd.GeoDataFrame(
        gdf[["longitude", "latitude"]].copy(),
        geometry=gpd.points_from_xy(gdf["longitude"], gdf["latitude"]),
        crs="EPSG:4326"
    ).to_crs(gdf.crs)
    joined = gpd.sjoin(points, ua[["geometry"]], how="left", predicate="within")
    gdf = gdf.copy()
    gdf["in_urban_area_2020"] = joined["index_right"].notna().values
    return gdf


In [17]:
facs_with_pop = add_urban_area_flag(
    facs_with_pop,
    "data_sources/cb_2020_us_ua20_500k.zip"
)


## Add Wildland-Urban Interface (WUI) type

In [ ]:
def add_wui_type(gdf, wui_path):
    """
    Adds wui_type: the CalFire Wildland-Urban Interface classification
    for each facility — "Intermix", "Interface", or "Influence Zone".
    Blank if the facility falls outside all WUI polygons.

    Source: CalFire Wildland_Urban_Interface shapefile
    """
    wui = gpd.read_file(wui_path)[["WUI_DESC", "geometry"]].to_crs(gdf.crs)
    points = gpd.GeoDataFrame(
        gdf[["longitude", "latitude"]].copy(),
        geometry=gpd.points_from_xy(gdf["longitude"], gdf["latitude"]),
        crs="EPSG:4326"
    ).to_crs(gdf.crs)
    joined = gpd.sjoin(points, wui, how="left", predicate="within")
    gdf = gdf.copy()
    gdf["wui_type"] = joined["WUI_DESC"].values
    return gdf


In [ ]:
facs_with_pop = add_wui_type(
    facs_with_pop,
    "data_sources/hazards/Wildland_Urban_Interface.zip"
)


## Export

In [18]:
facs_with_pop.to_csv("data_sources/facilities/ca_facilities.csv", index=False)

In [19]:
cdcr_facilities = facs_with_pop[facs_with_pop["type"] == "STATE"]

In [21]:
len(facs_with_pop), len(cdcr_facilities)

(357, 84)